# Suite BLD — Schema builders as visitors of their meta-schemas

`Builders` holds the meta-schemas (`Builders.Meta`), in which every schema object is a reference object and every
reference between schema objects is a relation, and the builders over them: each a writable `Visitors.OfObject`, with
a member for each property and adjacency of its meta-schema, and VISITOR_PATTERN.md's shorthands. Schema objects and
data are built in a store, `store.moduleName.[...moduleName].schemaName(...)`. See docs/META_SCHEMAS.md, Plan.

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders, members and the store are `Proxy`
objects whose members are typed loosely, so callbacks are annotated `(t: any)`; a name starting with `_` reads as
`undefined` on a visitor (Python raises `AttributeError`), while the store and its paths throw `AttributeError`; ints
are `bigint`s, a map is a `Map` or a record, and a container's size is `length`.

In [ ]:
import { Builders as B, Paths, Plain, Schemas as S } from "@mbse/schemas/Framework";
import { AttributeError, LookupError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, equal, map, raises, same } from "./support.js";

/** crm.Person, who knows other persons since a year, built in `store`, typed by its basic scalars. */
function person(store: any): [any, any] {
  const [Text, Int] = [store.Schemas.OfStr, store.Schemas.OfInt];
  const Person = store.Schemas.OfObject((o: any) => (
    o
      .name("crm.Person")
      .ref(true)
      .properties([
        (t: any) => t.name("given_name").schema([Text]),
        (t: any) => t.name("age").schema([Int]),
      ])
  ));
  const Knows = store.Schemas.OfRelation((r: any) => (
    r
      .name("crm.Knows")
      .links([
        (t: any) => t.name("knower"),
        (t: any) => t.name("known"),
      ])
      .properties([(t: any) => t.name("since").schema([Int])])
      .uniques("since")
  ));
  store.Schemas.OfObject(Person, (o: any) => o.adjacencies([
    (t: any) => t.name("acquaintances").relation([Knows]).me("knower"),
    (t: any) => t.relation([Knows]).me("known"),
  ]));
  return [Person, Knows];
}

/** `unique` clauses, each a sorted list of names, as Python compares frozensets. */
const clauses = (uniques: readonly ReadonlySet<string>[]) => uniques.map((unique) => [...unique].sort());

/** Python's `next(it)`. */
const next = (it: Iterator<any>) => it.next().value;

## BLD-01 · The meta-schemas: reference object schemas, and relations between them, each valid

In [ ]:
{
  const plain: any = new B.Store(); // a store is made holding its own plain schemas of schemas, and its own basic scalars
  for (const name of plain.names().filter((n: string) => n.startsWith("Schemas."))) {
    assert(plain.registered(name).validate().length === 0, `${name}: ${plain.registered(name).validate()}`);
  }
  const [OfObject, HasProperty, HasSchema] = [plain.Schemas.OfObject, plain.Schemas.HasProperty, plain.Schemas.HasSchema]
    .map((path: any) => path.as_schema());
  assert(OfObject.ref && same([...OfObject.properties.keys()], ["name", "description", "ref", "singleton"]));
  assert(same([...OfObject.adjacencies.keys()], ["parameters", "adjacencies", "properties", "refines"]));
  assert(same([...HasProperty.entry.properties.keys()], ["name"])); // a member's name is its entry's
  assert(equal(clauses(HasProperty.uniques).sort(), [["name", "owner"], ["property"]])); // one owner, one name
  assert(equal(clauses(HasSchema.uniques), [["schema"]]) && plain.Schemas.AnyOf.as_schema().uniques.length === 0);
  assert([...HasSchema.entry.links.values()].every((link: any) => link.schema === null)); // follows from the adjacencies
  assert(plain.Schemas.OfUnion.as_schema().anonymous.some((a: any) => a.relation?.name === "Schemas.HasSchema" && a.me === "schema")); // a type
  assert(!plain.Schemas.OfUnion.as_schema().adjacencies.has("refines")); // only scalars, objects, containers refine
  assert(!plain.Schemas.OfRelation.as_schema().anonymous.some((a: any) => a.relation?.name === "Schemas.HasSchema")); // not a type
  const uniques = plain.Schemas.OfRelation.as_schema().properties.get("uniques")?.schema as S.OfContainer.Data;
  assert((uniques.element as S.OfContainer.Data).element !== null);
  const other: any = new B.Store();
  assert(other.Schemas.OfObject.as_schema() !== OfObject && other.Schemas.OfInt.as_schema() !== plain.Schemas.OfInt.as_schema());
}

## BLD-02 · Building in a store: a callback creates, an object and a callback update, an object alone clones

In [ ]:
{
  const store: any = new B.Store();
  const [Person, Knows] = person(store);
  assert(store.registered("crm.Person") === Person && store.registered("crm.Knows") === Knows);
  assert(store.registered("Schemas.OfObject") === store.Schemas.OfObject.as_schema()); // the store holds its plain schemas of schemas
  assert(Person.validate().length === 0 && Knows.validate().length === 0);
  const bob = store.crm.Person((c: any) => c.given_name("Bob").age(41n));
  const ada = store.crm.Person((c: any) => (
    c
      .given_name("Ada")
      .age(36n)
      .acquaintances((adv: any) => adv.insert((env: any) => env.known(bob).since(2020n)))
  ));
  assert(same([...ada.acquaintances].map((e: any) => String(e.known.given_name)), ["Bob"]));
  store.crm.Person(ada, (c: any) => c.age(37n));
  assert(ada.age.get() === 37n);
  const twin = store.crm.Person(ada);
  assert(twin !== ada && String(twin.given_name) === "Ada");
  assert(repr(store.crm.Person) === "<crm.Person>");
  raises(TypeError, () => store.crm.Person(), "takes a callback, an object and a callback, or an object");
  raises(AttributeError, () => store.crm.Nobody((c: any) => c), "no schema registered as 'crm.Nobody'");
  raises(AttributeError, () => store._hidden);
  raises(AttributeError, () => store.crm._hidden);
  // JavaScript's protocol probes are not names: awaiting or inspecting a store or a path is safe
  assert(store.then === undefined && store.crm.then === undefined && store.crm[Symbol.toStringTag] === undefined);
}

## BLD-03 · Paths: a named schema's is its name, a member's its owner's then its name; one with no name of its own has none, and one object alone links it

In [ ]:
{
  const store: any = new B.Store();
  const [Text, Int] = [store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema()];
  const [Person] = person(store);
  /** A schema object's path as plain data: from the root module, ".", by name (Object identity, Decided, 81). */
  const path = (schemaObject: unknown): any => Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(schemaObject));
  const step = (name: string, i = "schemas", then: unknown = null): any =>
    map({ entry: { i, o: "", properties: { name }, next: then ?? { target: {} } } });
  const crmPerson = map({ singleton: ".", next: step("crm", "schemas", step("Person")) });
  assert(equal(path(Person), crmPerson) && store.held(Paths.of(Person)) === Person && store.held(crmPerson) === Person);
  assert(equal(path(store.Schemas.HasSchema.as_schema()), map({ singleton: ".", next: step("Schemas", "schemas", step("HasSchema")) })));
  const age = Person.properties.get("age");
  assert(equal(path(age), map({ singleton: ".", next: step("crm", "schemas", step("Person", "schemas", step("age", "properties"))) })));
  assert(store.held(path(age)) === age); // a member's: its owner's, then the adjacency that owns it and its name there
  assert(store.held(map({ singleton: ".", next: step("crm", "schemas", step("Person", "schemas", step("age", "nope"))) })) === null);
  assert(store.held(map({ singleton: "crm.Person", next: { target: {} } })) === null); // data's, not a schema object's
  raises(ValueError, () => store.Schemas.OfObject((o: any) => o.name("crm.Person")), "schema 'crm.Person' is already registered");
  store.Schemas.OfObject(Person, (o: any) => o.name("crm.Human")); // renamed: registered under its new name, at its new path
  assert(store.registered("crm.Human") === Person && !store.names().includes("crm.Person") && store.held(crmPerson) === null);
  assert(equal(path(Person), map({ singleton: ".", next: step("crm", "schemas", step("Human")) })));
  assert(store.crm.Human((c: any) => c.given_name("Cy")).as_schema() === Person);
  raises(ValueError, () => store.Schemas.OfObject((o: any) => o.name("crm.Human.v2")), // its module would be at the schema's path
    "the path 'crm.Human' names another schema object already: a module and a schema, or two schemas, do not share a path");
  // A schema object with no name of its own has no path, nor has one no store holds; one object alone links it.
  const Tags = store.Schemas.OfContainer((c: any) => c.index([Int]).element([Text]));
  raises(LookupError, () => Paths.of(Tags), "a container schema with no name of its own, nor a member of one that has a name, "
    + "has no path: it is written nested in the one object that links it");
  raises(LookupError, () => Paths.of(new B.OfProperty.Builder().schema([Text]).create()),
    "a property no store holds has no path: a schema object's path is in its store");
  const Tagged = store.Schemas.OfObject((o: any) => o.name("crm.Tagged").properties([(t: any) => t.name("tags").schema([Tags])]));
  raises(LookupError, () => Paths.of(Tags), "has no path"); // linked, but not owned: still none
  raises(ValueError, () => store.Schemas.OfObject((o: any) => o.name("crm.Labelled").properties([(t: any) => t.name("labels").schema([Tags])])),
    "a container schema with no name of its own is linked by one schema object alone, and another links it already: name it to share it");
  store.Schemas.OfProperty(Tagged.properties.get("tags"), (p: any) => p.schema([Text])); // unlinked: another may link it
  const Labelled = store.Schemas.OfObject((o: any) => o.name("crm.Labelled").properties([(t: any) => t.name("labels").schema([Tags])]));
  assert(Labelled.properties.get("labels").schema === Tags);
  const twin = new B.OfObject.Builder(Labelled, true, store).name("crm.Twin").clone(); // a clone has its own of what has no name
  assert(twin.properties.get("labels").schema !== Tags && twin.properties.get("labels").schema.element === Text);
}

## BLD-04 · A schema's members: the peer shorthand, each item linked, or a callback naming the entry and writing its member

In [ ]:
{
  const store: any = new B.Store();
  const Text = store.Schemas.OfStr.as_schema();
  const Int = store.Schemas.OfInt.as_schema();
  const spare = store.Schemas.OfProperty((p: any) => p.schema([Int]));
  const Contact = store.Schemas.OfObject((o: any) => (
    o
      .name("crm.Contact")
      .ref(true)
      .properties([
        (t: any) => t.name("given_name").schema([Text]),
        (t: any) => t.name("spare").property(spare).description("extra"), // linked, then updated
      ])
  ));
  assert(same([...Contact.properties.keys()], ["given_name", "spare"]) && Contact.properties.get("spare") === spare);
  assert(spare.description === "extra" && spare.name === "spare");
  raises(ValueError, () => new B.OfObject.Builder().properties([(t: any) => t.description("x").property(spare)]),
    "link the object before writing its members");
  const Bare = new B.OfObject.Builder().properties([(t: any) => t.name("x").property]).create(); // writes nothing: an empty one
  assert(same([...Bare.properties.keys()], ["x"]) && Bare.properties.get("x").schema === null);
  // the callback form: an entry, its name, and its link
  const Person = (
    new B.OfObject.Builder()
      .name("crm.Person")
      .ref(true)
      .properties((adv: any) => adv.insert((env: any) => env.name("nickname").property((p: any) => p.schema([Text]))))
      .create()
  );
  assert(Person.validate().length === 0 && Person.properties.get("nickname").schema === Text);
  raises(ValueError, () => new B.OfObject.Builder().properties((adv: any) => adv.insert((env: any) => env.name("x"))),
    "link 'property' is not set");
  raises(TypeError, () => new B.OfObject.Builder().properties((adv: any) => adv.insert(3n)),
    "an entry, or a callback taking one, is expected");
  raises(TypeError, () => new B.OfAny.Builder().schemas(new Map([["n", Int]])), "takes an iterable of what it links");
  raises(TypeError, () => new B.OfObject.Builder().properties([3n]), "link 'property' cannot link 3");
}

## BLD-05 · Relations: links by name, their schemas following from adjacencies; `me` and `uniques` naming links and properties

In [ ]:
{
  const store: any = new B.Store();
  const [Person, Knows] = person(store);
  assert(same([...Knows.entry.links.keys()], ["knower", "known"]) && equal(clauses(Knows.uniques), [["since"]]));
  assert(Person.adjacencies.get("acquaintances").me === "knower" && same(Person.anonymous.map((a: any) => a.me), ["known"]));
  for (const [given, held] of [
    ["since", [["since"]]],
    [["knower", "since"], [["knower", "since"]]],
    [[["since"], ["knower", "known"]], [["since"], ["knower", "known"]]],
    [["since", ["knower", "known"]], [["since"], ["knower", "known"]]],
    [null, []],
  ] as [unknown, string[][]][]) {
    assert(equal(clauses(new B.OfRelation.Builder(Knows).uniques(given).clone().uniques), held));
  }
  const builder = new B.OfRelation.Builder(Knows).uniques([["since"], ["known", "knower"], ["elsewhere"]]);
  assert(equal(builder.uniques._held(), [["since"], ["knower", "known"], ["elsewhere"]]) && builder.uniques.get() === null);
  assert(new B.OfRelation.Builder().uniques._held() === null);
  const found: unknown[] = [];
  const dicts: unknown[] = [];
  new B.OfRelation.Builder(Knows).uniques.as_value((v: any) => v.as_container((c: any) => (
    c.as_dict((d: unknown) => dicts.push(d)),
    c.as_entries((it: Iterable<[unknown, any]>) => [...it].map(([, e]) => e.as_container((n: any) => n.as_entries(
      (names: Iterable<[unknown, any]>) => found.push([...names].map(([, x]) => x._held()))))))
  )));
  assert(equal(found, [["since"]]) && dicts.length === 0); // by position: as_dict is not called
  const widened = new B.OfRelation.Builder(Knows);
  widened.uniques.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_container((n: any) => n.at(1n).as_scalar("knower"))));
  assert(equal(clauses(widened.clone().uniques), [["knower", "since"]]));
  widened.uniques.as_value((v: any) => v.as_container((c: any) => c.remove(0n)));
  assert(widened.uniques._held() === null);
  const Unrelated = new B.OfAdjacency.Builder(Person.adjacencies.get("acquaintances"))
    .relation((adv: any) => adv.remove((env: any) => env.relation((r: any) => r.uniques("since")))).clone();
  assert(Unrelated.relation === null); // the template's uniques agree with the relation's by value
}

## BLD-06 · Property shorthands: a value whole, of exactly the property's schema; a container's through its visitor

In [ ]:
{
  for (const wrong of [() => new B.OfObject.Builder().ref(1n), () => new B.OfObject.Builder().name(3n)]) {
    raises(TypeError, wrong, "holds");
  }
  // A property a map holds, as an extension may add: set whole, or visited as a container.
  const attributed: any = new B.Store().extend((m: any) => m.Schemas.OfObject(m.Schemas.OfPattern, (o: any) => o.properties([
    (t: any) => t.name("attributes").schema([m.Schemas.OfContainer((c: any) => c.index([m.Schemas.OfStr]).element([
      m.Schemas.OfAny((a: any) => a.schemas([m.Schemas.OfStr, m.Schemas.OfInt, m.Schemas.OfFloat]))]))]),
    (t: any) => t.name("default").schema([m.Schemas.OfAny((a: any) => a.schemas([m.Schemas.OfStr, m.Schemas.OfInt]))]),
  ])));
  const called = attributed.Schemas.OfPattern((m: any) => m.name("call").attributes({ name: "knows" })
    .attributes(new Map([["arity", 2n]])).default(3n));
  assert(new B.OfPattern.Builder(called, false, attributed).default._held() === 3n); // a value of any schema: as given
  assert(equal(new B.OfPattern.Builder(called, false, attributed).attributes._held(), map({ arity: 2n }))); // set whole: nothing merges; a map is a Map or a record
  raises(TypeError, () => attributed.Schemas.OfPattern((m: any) => m.attributes(["x"])), "holds a map");
  raises(TypeError, () => new B.OfRelation.Builder().uniques._matched("since"), "holds a list");
  raises(TypeError, () => new B.OfRelation.Builder().uniques._matched(3n), "holds a list");
  const pattern = new B.OfPattern.Builder(null, true, attributed).name("call"); // in the store, which holds its plain schema
  pattern.attributes((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at("name").as_scalar("knows"))));
  pattern.attributes.as_value((v: any) => v.as_container((c: any) => c.at("arity").as_scalar((s: any) => s.set(2n))));
  const seen = new Map<string, unknown>();
  pattern.attributes.as_value((v: any) => v.as_container((c: any) => (
    seen.set("length", c.length).set("schema", c.as_schema()),
    c.as_dict((d: Map<unknown, unknown>) => seen.set("keys", [...d.keys()])),
    c.as_entries((it: Iterable<[unknown, any]>) => seen.set("values", [...it].map(([, e]) => e._held()))),
    c.as_ordered(() => seen.set("ordered", true))
  )));
  assert(seen.get("length") === 2 && equal(seen.get("keys"), ["name", "arity"]) && equal(seen.get("values"), ["knows", 2n]));
  assert(!seen.has("ordered")
    && seen.get("schema") === attributed.Schemas.OfPattern.as_schema().properties.get("attributes")?.schema);
  pattern.attributes.as_value((v: any) => v.as_container((c: any) => c.remove("arity")));
  pattern.attributes.as_value((v: any) => v.as_container({ name: "calls" }));
  pattern.attributes.as_value((v: any) => v.as_container([["name", "calls"]])); // a map given as its entries
  assert(equal(pattern.attributes._held(), map({ name: "calls" })));
  pattern.attributes.as_value((v: any) => v.clear());
  assert(pattern.attributes._held() === null);
  pattern.attributes.as_value((v: any) => v.as_container((c: any) => c)); // selecting the kind writes it, empty
  assert(equal(pattern.attributes._held(), map({})));
  pattern.name.as_value((v: any) => (v.as_scalar((s: any) => (s.set("let"), s.get(), s.as_schema())), v.as_schema()));
  assert(String(pattern.name) === "let" && pattern.name.get() === "let");
  pattern.name.as_value((v: any) => v.as_scalar((s: any) => s.clear()));
  assert(pattern.name.get() === null);
  pattern.name.as_scalar("var").as_value((v: any) => v.clear());
  assert(pattern.name.get() === null);
  raises(TypeError, () => pattern.name.as_value((v: any) => v.as_object({})), "holds a scalar");
  raises(TypeError, () => pattern.attributes.as_value((v: any) => v.as_scalar(1n)), "holds a container");
  const reasons: unknown[] = [];
  pattern.name.as_object((o: any) => reasons.push(o.get()));
  pattern.attributes.as_value((v: any) => v.as_scalar((s: any) => reasons.push(s.get())));
  assert(same(reasons, [null, null]) && repr(pattern.name) === "<property 'name'>");
  // A template agrees with a member's map by value; an OfAny's unnamed types take identities their path derives.
  const tagged: any = new B.Store().extend((m: any) => m.Schemas.OfObject(m.Schemas.OfProperty, (o: any) => o.properties([
    (t: any) => t.name("tags").schema([m.Schemas.OfContainer((c: any) => c.index([m.Schemas.OfStr]).element([m.Schemas.OfStr]))])])));
  const Tagged = tagged.Schemas.OfObject((o: any) => o.name("demo.Tagged").properties([
    (t: any) => t.name("a").schema([tagged.Schemas.OfStr]).tags({ unit: "m" }),
    (t: any) => t.name("b").schema([tagged.Schemas.OfStr]).tags({ unit: "s" }),
    (t: any) => t.name("v").schema([tagged.Schemas.OfAny((a: any) => a.schemas([
      tagged.Schemas.OfContainer((c: any) => c.index([tagged.Schemas.OfInt]).element([tagged.Schemas.OfStr]))]))]),
  ]));
  tagged.Schemas.OfObject(Tagged, (o: any) => o.properties((adv: any) => adv.remove((env: any) => env.property((p: any) => p.tags({ unit: "s" })))));
  assert(same([...Tagged.properties.keys()], ["a", "v"]));
  assert(tagged.held(Paths.of(Tagged.properties.get("v"))) === Tagged.properties.get("v")); // its path its owner's, then its name
}

## BLD-07 · Adjacency shorthands: one entry replaced where the relation allows one; `insert` does not replace

In [ ]:
{
  const store: any = new B.Store(); // the basic scalars below are its own
  const Text = store.Schemas.OfStr.as_schema();
  const Int = store.Schemas.OfInt.as_schema();
  const Names = (
    new B.OfContainer.Builder()
      .index([Int])
      .element([Text])
      .layout("dense")
      .create()
  );
  assert(Names.validate().length === 0 && Names.arrangement === "dense");
  const Keyed = new B.OfContainer.Builder(Names).index([Text]).clone();
  assert(Keyed.index === Text && Names.index === Int);
  raises(ValueError, () => new B.OfContainer.Builder(Names).index((adv: any) => adv.insert((env: any) => env.index(Text))),
    "remove it before inserting another");
  new B.OfContainer.Builder(Names).index((adv: any) => adv.insert((env: any) => env.index(Int))); // held already: elided
  raises(TypeError, () => new B.OfProperty.Builder().schema((adv: any) => adv.insert((env: any) => env.schema((s: any) => s))),
    "links an object that exists");
  raises(TypeError, () => new B.OfProperty.Builder().schema([(t: any) => t.description("x")]),
    "links an object that exists: link it, not its members");
  raises(TypeError, () => new B.OfProperty.Builder().schema((adv: any) => adv.insert((env: any) => env.schema(3n))),
    "link 'schema' cannot link 3");
  const cleared = new B.OfContainer.Builder(Names).index((adv: any) => adv.remove((env: any) => env.index(Int))).clone();
  assert(cleared.index === null);
}

## BLD-08 · Reading back: members are child visitors; a link reads its referent, read-only

In [ ]:
{
  const store: any = new B.Store();
  const Text = store.Schemas.OfStr.as_schema();
  const [Person] = person(store);
  const builder = new B.OfObject.Builder(Person, true, store); // in the store, which holds its plain schema
  assert(builder.name.get() === "crm.Person" && builder.ref.get() === true && builder.name.as_schema().name === "name");
  assert(same([...builder.properties].map((env: any) => String(env.name)), ["given_name", "age"]));
  const first = [...builder.properties][0];
  assert(first.name.as_schema() === store.Schemas.HasProperty.as_schema().entry.properties.get("name") && first.as_schema() === store.Schemas.HasProperty.as_schema().entry);
  assert(first.property.as_schema() === store.Schemas.OfProperty.as_schema());
  assert(first.property.schema.as_schema().relation === store.Schemas.HasSchema.as_schema()
    && same([...first.property.as_properties].map((p: any) => p.as_schema().name), []));
  assert(repr(first) === "<entry of 'properties'>" && repr(first.property) === "<link 'property'>");
  assert(repr(builder.properties) === "<adjacency 'properties'>" && repr(builder) === "<builder of an object schema>");
  const links: string[] = [];
  first.property.as_adjacencies((adv: any) => links.push(adv.as_schema().name));
  assert(same(links, ["schema"]));
  assert(builder.as_schema() === store.Schemas.OfObject.as_schema() && builder.properties.as_schema() === store.Schemas.OfObject.as_schema().adjacencies.get("properties"));
  assert(same([...builder.as_properties].map((p: any) => p.as_schema().name), ["name", "ref"]));
  const seen: string[] = [];
  builder.as_adjacencies((adv: any) => seen.push(adv.as_schema().name));
  assert(same(seen, ["parameters", "adjacencies", "properties", "refines"]));
  const iterated: string[] = [];
  builder.properties.as_iterable((it: Iterator<any>) => iterated.push(...[...{ [Symbol.iterator]: () => it }].map((e) => String(e.name))));
  assert(same(iterated, ["given_name", "age"]));
  raises(ReadOnlyError, () => first.property.description("x"), "read-only");
  raises(ReadOnlyError, () => builder.nonesuch("x"));
  assert(first.nonesuch.get() === null && first.property._target() !== null);
  new B.OfObject.Builder().properties((adv: any) => adv.insert((env: any) => (env.property.as_schema(), env.property((p: any) => p))));
  assert(new B.OfAny.Builder(null, true, store).schemas.as_schema() === store.Schemas.OfAny.as_schema().adjacencies.get("schemas"));
  B.OfProperty.Builder._meta = "Nonesuch";
  const detached = new B.OfProperty.Builder();
  assert(detached.description.as_schema() === null && detached.schema.as_schema() === null);
  B.OfProperty.Builder._meta = "OfProperty";
  B.OfObject.Builder._meta = "Nonesuch";
  const unschemed: unknown[] = [];
  new B.OfObject.Builder().properties([(t: any) => unschemed.push(t.as_schema(), t.name.as_schema())]);
  B.OfObject.Builder._meta = "OfObject";
  assert(same(unschemed, [null, null]));
  const shared = new B.OfObject.Builder().properties((adv: any) => adv.insert((env: any) => env.property(first.property))).create();
  assert(shared.properties.get("given_name") === Person.properties.get("given_name")); // linked through the link given
  assert(builder._hidden === undefined && first._hidden === undefined && first.property._hidden === undefined); // Python: AttributeError
  raises(TypeError, () => new B.OfObject.Builder(Text), "is given");
}

## BLD-09 · Editing through an entry: its member updated, or renamed; removing by entry or by template

In [ ]:
{
  const store: any = new B.Store();
  const Int = store.Schemas.OfInt.as_schema();
  const [Person] = person(store);
  const age = Person.properties.get("age");
  store.Schemas.OfObject(Person, (o: any) => o.properties((adv: any) => adv.as_iterable(
    (it: Iterator<any>) => (next(it), next(it).property((p: any) => p.description("years"))))));
  assert(Person.properties.get("age") === age && age.description === "years");
  store.Schemas.OfObject(Person, (o: any) => o.properties((adv: any) => adv.as_iterable(
    (it: Iterator<any>) => (next(it), next(it).property((p: any) => p))))); // writes nothing: nothing changes
  store.Schemas.OfObject(Person, (o: any) => o.properties((adv: any) => adv.as_iterable((it: Iterator<any>) => next(it).name("first_name"))));
  assert(same([...Person.properties.keys()], ["first_name", "age"]) && Person.validate().length === 0);
  store.Schemas.OfObject(Person, (o: any) => o.properties((adv: any) => adv.as_iterable(
    (it: Iterator<any>) => next(it).name.as_scalar("given"))));
  assert(same([...Person.properties.keys()], ["given", "age"]));
  raises(TypeError, () => new B.OfObject.Builder(Person).properties((adv: any) => adv.as_iterable(
    (it: Iterator<any>) => next(it).name(3n))), "a name is non-empty text");
  raises(TypeError, () => new B.OfObject.Builder(Person).properties([(t: any) => t.name("")]), "a name is non-empty text");
  const Smaller = (
    new B.OfObject.Builder(Person)
      .name("crm.Smaller")
      .properties((adv: any) => adv.remove((env: any) => env.name("age")))
      .clone()
  );
  assert(same([...Smaller.properties.keys()], ["given"]) && same([...Person.properties.keys()], ["given", "age"]));
  const held = [...new B.OfObject.Builder(Person).properties][0];
  const Fewer = new B.OfObject.Builder(Person).properties((adv: any) => adv.remove(held)).clone();
  assert(same([...Fewer.properties.keys()], ["age"]));
  const Typed = new B.OfObject.Builder(Person).properties((adv: any) => adv.remove(
    (env: any) => env.property((p: any) => p.schema([Int])))).clone();
  assert(same([...Typed.properties.keys()], ["given"]));
  raises(TypeError, () => new B.OfObject.Builder(Person).properties((adv: any) => adv.remove(3n)),
    "an entry, or a callback taking one, is expected");
  const Copied = new B.OfObject.Builder().name("demo.Copied").properties(new B.OfObject.Builder(Person).properties).create();
  assert(same([...Copied.properties.keys()], ["given", "age"]));
  const Again = new B.OfObject.Builder().name("demo.Again").properties((adv: any) => adv.insert(held)).create();
  assert(same([...Again.properties.keys()], ["given"]));
}

## BLD-10 · A clone owns its members, cloned with paths of their own, and its own of what has no name; inherited members stay shared

In [ ]:
{
  const store: any = new B.Store();
  const Text = store.Schemas.OfStr.as_schema();
  const Int = store.Schemas.OfInt.as_schema();
  const [Person] = person(store);
  const Twin = store.Schemas.OfObject(new B.OfObject.Builder(Person).name("crm.Twin").clone());
  assert(Twin !== Person && same([...Twin.properties.keys()], ["given_name", "age"]));
  assert(Twin.properties.get("age") !== Person.properties.get("age") && Twin.properties.get("age").schema === Int);
  assert(!Paths.same(Paths.of(Twin.properties.get("age")), Paths.of(Person.properties.get("age")))); // each its owner's
  const Employee = store.Schemas.OfObject((o: any) => (
    o
      .name("crm.Employee")
      .refines([Person])
      .properties([(t: any) => t.name("badge").schema([Text])])
  ));
  assert(Employee.ref && same([...Employee.properties.keys()], ["given_name", "age", "badge"]));
  const Copy = new B.OfObject.Builder(Employee).name("crm.Copy").clone();
  assert(Copy.properties.get("age") === Person.properties.get("age")
    && Copy.properties.get("badge") !== Employee.properties.get("badge"));
  assert(Copy.adjacencies.get("acquaintances") === Person.adjacencies.get("acquaintances") && same(Copy.anonymous, Person.anonymous));
}

## BLD-11 · Every kind: refinement, parameters, containers, unions, intersections, values of any schema, patterns

In [ ]:
{
  const store: any = new B.Store();
  const Text = store.Schemas.OfStr.as_schema();
  const Int = store.Schemas.OfInt.as_schema();
  const width = store.Schemas.OfParameter((p: any) => p.description("Its width"));
  const Sized = store.Schemas.OfObject((o: any) => (
    o
      .name("demo.Sized")
      .parameters([(t: any) => t.name("width").parameter(width)])
  ));
  assert(Sized.parameters.get("width") === width && width.name === "width");
  assert(new B.OfScalar.Builder(Int).name.get() === "Schemas.OfInt" && new B.OfScalar.Builder().name.get() === null);
  const Names = store.Schemas.OfContainer((c: any) => c.name("demo.Names").index([Int]).element([Text]));
  const Tags = store.Schemas.OfContainer((c: any) => c.name("demo.Tags").refines([Names]));
  assert(Tags.index === Int && Tags.element === Text && Tags.validate().length === 0);
  const Narrow = store.Schemas.OfContainer((c: any) => c.name("demo.Narrow").refines([Names]).index([Int]).element([Text]));
  assert(Narrow.index === Int && Narrow.element === Text && Narrow.validate().length === 0); // its own, given
  const Shape = store.Schemas.OfUnion((u: any) => (
    u
      .name("geo.Shape")
      .flat(true)
      .branches([
        (t: any) => t.name("count").schema([Int]),
        (t: any) => t.name("label").schema([Text]),
      ])
  ));
  const Stamped = store.Schemas.OfIntersection((i: any) => (
    i
      .name("geo.Stamped")
      .parts([
        (t: any) => t.name("area").schema([store.Schemas.OfObject((o: any) => o.name("geo.Area").properties([
          (p: any) => p.name("size").schema([Int]),
        ]))]),
        (t: any) => t.name("when").schema([store.Schemas.OfObject((o: any) => o.name("geo.When").properties([
          (p: any) => p.name("at").schema([Int]),
        ]))]),
      ])
  ));
  const Either = store.Schemas.OfAny((a: any) => a.schemas([Int, Text]));
  const Mutual = store.Schemas.OfPattern((p: any) => (
    p
      .name("crm.Mutual")
      .symbols([
        (t: any) => t.name("a").schema([store.Schemas.OfObject((o: any) => o.name("crm.Who").ref(true))]),
        (t: any) => t.name("b").schema([store.registered("crm.Who")]),
      ])
  ));
  for (const schema of [Sized, Shape, Stamped, Either, Mutual]) {
    assert(schema.validate().length === 0, `${schema.name}: ${schema.validate()}`);
  }
  assert(same(Shape.branches.map((b: any) => b.name), ["count", "label"]) && same([...Mutual.symbols.keys()], ["a", "b"]));
  assert(same(Either.schemas, [Int, Text]));
  const Linked = new B.OfAdjacency.Builder().relation((adv: any) => adv.insert((env: any) => env.relation((r: any) => r
    .name("demo.R").links([(t: any) => t.name("a"), (t: any) => t.name("b")])))).me("a").create();
  assert(Linked.relation.name === "demo.R" && Linked.me === "a");
}

## BLD-12 · Edges: reading through the shorthand's visitor, a scalar's conversions, finalizing in the wrong order

In [ ]:
{
  const store: any = new B.Store(); // the basic scalars below are its own
  const Int = store.Schemas.OfInt.as_schema();
  const scalar = new B.OfScalar.Builder(null, true, store).name("demo.Name"); // in the store, which holds its plain schema
  raises(TypeError, () => scalar.name.as_value((v: any) => v.as_container({})), "holds a scalar");
  raises(TypeError, () => scalar.name.as_value((v: any) => v.as_scalar((s: any) => Number(s))), "property 'name' holds a str, not an int");
  const Names = new B.OfContainer.Builder().name("demo.Names").index([Int]).element([Int]).create();
  const Narrowed = new B.OfContainer.Builder().name("demo.Narrowed").refines([Names]).create();
  assert(new B.OfContainer.Builder(Narrowed).refines([Names]).clone().refines === Names); // held already: elided
  const seen: unknown[] = [];
  new B.OfObject.Builder(null, true, store).properties([(t: any) => (
    seen.push([t.as_schema(), t.name.get()]),
    t.name((n: any) => n.as_scalar("x")),
    seen.push([t.name.get(), t.name.as_schema().name])
  )]);
  assert(equal(seen, [[store.Schemas.HasProperty.as_schema().entry, null], ["x", "name"]]));
  const hidden: unknown[] = [];
  new B.OfObject.Builder().properties([(t: any) => hidden.push(t._hidden, t.name._hidden)]);
  assert(same(hidden, [undefined, undefined])); // Python: AttributeError
  raises(ValueError, () => new B.OfProperty.Builder().schema([(t: any) => t]), "link 'schema' is not set");
  raises(ValueError, () => new B.OfObject.Builder(new B.OfObject.Builder().create()).create(),
    "create() is only valid without a source instance");
  const twice = new B.OfObject.Builder().name("demo.Twice");
  const first = twice.create();
  const second = twice.description("later").create();
  assert(first !== second && first.description === null && second.description === "later");
  const again = new B.OfObject.Builder(first);
  assert(again.clone() !== again.clone() && first.description === null);
  raises(ValueError, () => new B.OfObject.Builder().clone(), "clone() is only valid with a source instance");
  raises(ValueError, () => new B.OfObject.Builder().update(), "update() is only valid with a source instance");
  raises(TypeError, () => new B.OfObject.Builder().create(1n), "create() takes no arguments (1 given)"); // checked explicitly
}

## BLD-13 · Proxies: a container property's value given whole; the peer shorthand on adjacencies, replacing where one entry is allowed

In [ ]:
{
  const store: any = new B.Store();
  const Text = store.Schemas.OfStr.as_schema();
  const Int = store.Schemas.OfInt.as_schema();
  const P = store.Schemas.OfObject((o: any) => (
    o
      .name("crm.P")
      .ref(true)
      .properties([
        (t: any) => t.name("n").schema([Text]),
        (t: any) => t.name("tags").schema([store.Schemas.OfContainer((c: any) => c.index([Int]).element([Text]))]),
        (t: any) => t.name("marks").schema([store.Schemas.OfContainer((c: any) => c.index([Text]).element([Int]))]),
      ])
  ));
  const K = store.Schemas.OfRelation((r: any) => (
    r
      .name("crm.K")
      .links([(t: any) => t.name("a"), (t: any) => t.name("b")])
      .properties([(t: any) => t.name("since").schema([Int])])
  ));
  store.Schemas.OfRelation((r: any) => r.name("crm.Best").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).uniques("b"));
  store.Schemas.OfRelation((r: any) => r.name("crm.Three").links([(t: any) => t.name("a"), (t: any) => t.name("b"),
    (t: any) => t.name("c")]));
  const [Best, Three] = [store.registered("crm.Best"), store.registered("crm.Three")];
  store.Schemas.OfObject(P, (o: any) => o.adjacencies([
    (t: any) => t.name("knows").relation([K]).me("a"),
    (t: any) => t.relation([K]).me("b"),
    (t: any) => t.name("best").relation([Best]).me("a"),
    (t: any) => t.relation([Best]).me("b"),
    (t: any) => t.name("trio").relation([Three]).me("a"),
    (t: any) => t.relation([Three]).me("b"),
    (t: any) => t.relation([Three]).me("c"),
  ]));
  const bob = store.crm.P((c: any) => c.n("Bob").tags(["x", "y"]).marks({ math: 3n }));
  const tags: unknown[] = [];
  const marks: unknown[] = [];
  bob.tags.as_value((v: any) => v.as_container((c: any) => c.as_entries((it: Iterable<[unknown, any]>) => [...it].map(
    ([, e]) => e.as_scalar((s: any) => tags.push(s.get()))))));
  bob.marks.as_value((v: any) => v.as_container((c: any) => c.as_entries((it: Iterable<[unknown, any]>) => [...it].map(
    ([i, e]) => e.as_scalar((s: any) => marks.push([i, s.get()]))))));
  assert(same(tags, ["x", "y"]) && equal(marks, [["math", 3n]]));
  const ada = store.crm.P((c: any) => c.n("Ada").knows([bob, (t: any) => t.since(2020n).n("Cy")]));
  assert(same([...ada.knows].map((e: any) => String(e.b.n)), ["Bob", "Cy"])
    && equal([...ada.knows].map((e: any) => e.since.get()), [null, 2020n]));
  store.crm.P(ada, (c: any) => c.knows([(t: any) => t.b(bob).n("Robert")]));
  assert(String(bob.n) === "Robert" && [...ada.knows].length === 3);
  store.crm.P(ada, (c: any) => c.best([bob]));
  store.crm.P(ada, (c: any) => c.best([(t: any) => t.n("Dee")]));
  assert(same([...ada.best].map((e: any) => String(e.b.n)), ["Dee"])); // one entry allowed: replaced
  store.crm.P(ada, (c: any) => c.knows([...ada.knows].slice(0, 1))); // the entry shorthand: inserted
  assert([...ada.knows].length === 4);
  raises(TypeError, () => store.crm.P(ada, (c: any) => c.trio([bob])), "have more than two links; give entries");
  raises(TypeError, () => store.crm.P(ada, (c: any) => c.knows(new Map([["x", bob]]))), "takes an iterable of entries or of what it links");
  raises(ValueError, () => store.crm.P((c: any) => c.knows([(t: any) => t.n("Ed").b(bob)])), "link the object before writing its members");
  raises(ValueError, () => store.crm.P((c: any) => c.knows([(t: any) => t.since(1n)])), "link 'b' is not set");
  const seen: unknown[] = [];
  store.crm.P((c: any) => c.knows([(t: any) => (seen.push([t.as_schema() === K.entry, t.since.as_schema().name]), t.n("Fay"))]));
  assert(equal(seen, [[true, "since"]]));
  const hidden: unknown[] = [];
  store.crm.P((c: any) => c.knows([(t: any) => (hidden.push(t._hidden, t.n._hidden), t.n("Ivy"))]));
  assert(same(hidden, [undefined, undefined])); // Python: AttributeError
  const chained: string[] = [];
  store.crm.P((c: any) => c.knows([(t: any) => (chained.push(t.n.as_schema().name), t.n("Gus"))]));
  assert(same(chained, ["n"]));
  const Trio = store.Schemas.OfRelation((r: any) => r.name("crm.Trio").links([(t: any) => t.name("a"), (t: any) => t.name("b"), (t: any) => t.name("c")]));
  store.Schemas.OfObject(P, (o: any) => o.adjacencies([
    (t: any) => t.name("anon").relation([Trio]).me("a"),
    (t: any) => t.relation([Trio]).me("b"),
    (t: any) => t.relation([Trio]).me("c"),
  ]));
  raises(TypeError, () => store.crm.P(ada, (c: any) => c.anon([bob])), "entries of crm.Trio have more than two links");
  const Lone = store.Schemas.OfRelation((r: any) => r.links([(t: any) => t.name("a"), (t: any) => t.name("b"), (t: any) => t.name("c")]));
  store.Schemas.OfObject(P, (o: any) => o.adjacencies([(t: any) => t.name("lone").relation([Lone]).me("a")])); // unnamed: one links it
  raises(TypeError, () => store.crm.P(ada, (c: any) => c.lone([bob])), "entries of its relation have more than two links");
  const shown: string[] = [];
  store.crm.P((c: any) => c.knows([(t: any) => (shown.push(repr(t)), t.n("Jo"))]));
  assert(same(shown, ["<_Them object>"])); // Python's default repr, less its module and address
  raises(TypeError, () => store.crm.P((c: any) => c.tags("xy")), "takes a container of its schema");
  const hal = store.crm.P((c: any) => c.knows([(t: any) => t.b((o: any) => o.n("Hal")).n("Hank")])); // built at the link, then written
  assert(same([...hal.knows].map((e: any) => String(e.b.n)), ["Hank"]));
}